## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 6: Explainability in Recommender Systems
## Task 13: Evaluating Explainability

Compare how different explainability methods affect user understanding.
- Do explanations make recommendations clearer?
- Do explanations reveal biases in recommendations?

**Methods compared:** SHAP (feature-based), Neighborhood (User-CF / Item-CF), LIME (model-agnostic)

**Dataset:** MovieLens ml-latest-small

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'shap', 'lime', 'matplotlib', '-q'])

import os, zipfile, urllib.request, re, warnings, time
import numpy as np
import pandas as pd
import shap
import lime.lime_tabular
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
warnings.filterwarnings('ignore')
print('All imports successful!')

### Step 1: Load Data and Prepare Models

In [ ]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))

# Movie features
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else np.nan
movies['year'] = movies['title'].apply(extract_year).fillna(1995)
all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
for genre in all_genres:
    movies[f'g_{genre}'] = movies['genres'].apply(lambda x, g=genre: 1.0 if g in x.split('|') else 0.0)
genre_cols = [f'g_{g}' for g in all_genres]
movie_avg = ratings.groupby('movieId')['rating'].mean()
movies['avg_rating'] = movies['movieId'].map(movie_avg).fillna(ratings['rating'].mean())
mid_to_title = dict(zip(movies['movieId'], movies['title']))

# User preferences
ratings_g = ratings.merge(movies[['movieId'] + genre_cols], on='movieId')
user_prefs = {}
for uid, group in ratings_g.groupby('userId'):
    prefs = []
    for g in genre_cols:
        mask = group[g] == 1.0
        prefs.append(group.loc[mask, 'rating'].mean() if mask.sum() > 0 else 0.0)
    user_prefs[uid] = prefs
user_feat = pd.DataFrame.from_dict(user_prefs, orient='index', columns=genre_cols)
movie_feat = movies.set_index('movieId')[genre_cols + ['year', 'avg_rating']]
print(f'Movies: {len(movies)}, Ratings: {len(ratings)}')

### Step 2: Train Content-Based Model and Prepare CF Matrices

In [ ]:
# --- Content-Based Model (for SHAP and LIME) ---
feature_names = [f'movie_{c}' for c in genre_cols + ['year', 'avg_rating']] + [f'user_{c}' for c in genre_cols]
feature_rows, labels = [], []
for _, row in ratings.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if mid not in movie_feat.index or uid not in user_feat.index: continue
    mf = movie_feat.loc[mid].values
    uf = user_feat.loc[uid].values
    feature_rows.append(np.concatenate([mf, uf]))
    labels.append(row['rating'])
X = np.array(feature_rows)
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cbf_model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
cbf_model.fit(X_train, y_train)
print(f'CBF Model RMSE: {np.sqrt(((cbf_model.predict(X_test) - y_test)**2).mean()):.4f}')

# --- CF Matrices ---
user_ids = sorted(ratings['userId'].unique())
movie_ids = sorted(ratings['movieId'].unique())
uid_map = {u: i for i, u in enumerate(user_ids)}
mid_map = {m: i for i, m in enumerate(movie_ids)}
idx_to_mid = {i: m for m, i in mid_map.items()}
rows_r = ratings['userId'].map(uid_map).values
cols_r = ratings['movieId'].map(mid_map).values
ui_dense = csr_matrix((ratings['rating'].values, (rows_r, cols_r)),
    shape=(len(user_ids), len(movie_ids))).toarray()
item_sim = cosine_similarity(ui_dense.T)
np.fill_diagonal(item_sim, 0)
user_sim = cosine_similarity(ui_dense)
np.fill_diagonal(user_sim, 0)
print('CF matrices ready.')

### Step 3: Define Explanation Methods

In [ ]:
# SHAP explainer
shap_explainer = shap.TreeExplainer(cbf_model)

# LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train, feature_names=feature_names, mode='regression', random_state=42)

def get_shap_explanation(instance, top_n=5):
    """Return top SHAP features for an instance."""
    sv = shap_explainer.shap_values(instance.reshape(1, -1))[0]
    feat_impact = sorted(zip(feature_names, sv), key=lambda x: abs(x[1]), reverse=True)
    return feat_impact[:top_n]

def get_lime_explanation(instance, top_n=5):
    """Return top LIME features for an instance."""
    exp = lime_explainer.explain_instance(instance, cbf_model.predict, num_features=top_n)
    return exp.as_list()

def get_item_cf_explanation(user_id, movie_id, k=5):
    """Return similar items the user liked."""
    if user_id not in uid_map or movie_id not in mid_map:
        return []
    uid_idx, mid_idx = uid_map[user_id], mid_map[movie_id]
    user_ratings = ui_dense[uid_idx]
    rated_items = np.where(user_ratings > 0)[0]
    if len(rated_items) == 0: return []
    sims = item_sim[mid_idx]
    item_data = [(idx, sims[idx], user_ratings[idx]) for idx in rated_items if sims[idx] > 0]
    item_data.sort(key=lambda x: x[1], reverse=True)
    return [(mid_to_title.get(idx_to_mid[i], '?'), sim, rat) for i, sim, rat in item_data[:k]]

def get_user_cf_explanation(user_id, movie_id, k=5):
    """Return similar users who rated this movie."""
    if user_id not in uid_map or movie_id not in mid_map:
        return []
    uid_idx, mid_idx = uid_map[user_id], mid_map[movie_id]
    sims = user_sim[uid_idx]
    raters = np.where(ui_dense[:, mid_idx] > 0)[0]
    rater_data = [(r, sims[r], ui_dense[r, mid_idx]) for r in raters if sims[r] > 0]
    rater_data.sort(key=lambda x: x[1], reverse=True)
    return rater_data[:k]

print('All explanation methods ready.')

### Step 4: Metric 1 — Explanation Fidelity

How well does the explanation approximate the model's actual behavior? Measure by checking if the top features identified by each method actually influence the prediction when perturbed.

In [ ]:
def measure_fidelity(instances, method='shap', top_n=5):
    """Measure explanation fidelity: how much does prediction change when top features are zeroed out?"""
    fidelity_scores = []
    for inst in instances:
        original_pred = cbf_model.predict(inst.reshape(1, -1))[0]

        if method == 'shap':
            feats = get_shap_explanation(inst, top_n)
            top_feat_indices = [feature_names.index(f) for f, _ in feats if f in feature_names]
        elif method == 'lime':
            feats = get_lime_explanation(inst, top_n)
            top_feat_indices = []
            for feat_str, _ in feats:
                for i, fn in enumerate(feature_names):
                    if fn in feat_str:
                        top_feat_indices.append(i)
                        break

        if not top_feat_indices:
            continue

        # Zero out top features and re-predict
        perturbed = inst.copy()
        perturbed[top_feat_indices] = 0
        perturbed_pred = cbf_model.predict(perturbed.reshape(1, -1))[0]
        fidelity_scores.append(abs(original_pred - perturbed_pred))

    return np.mean(fidelity_scores) if fidelity_scores else 0

# Sample instances
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=100, replace=False)
sample_instances = X_test[sample_idx]

shap_fidelity = measure_fidelity(sample_instances, method='shap', top_n=5)
lime_fidelity = measure_fidelity(sample_instances, method='lime', top_n=5)

print('Explanation Fidelity (higher = explanation captures more important features):')
print(f'  SHAP: {shap_fidelity:.4f} avg prediction change when top-5 features removed')
print(f'  LIME: {lime_fidelity:.4f} avg prediction change when top-5 features removed')

### Step 5: Metric 2 — Explanation Stability

How consistent are explanations across multiple runs?

In [ ]:
def measure_stability(instance, method='shap', n_runs=10, top_n=5):
    """Measure how stable the top-N features are across runs."""
    all_top_features = []
    for _ in range(n_runs):
        if method == 'shap':
            feats = get_shap_explanation(instance, top_n)
            top = set(f for f, _ in feats)
        elif method == 'lime':
            feats = get_lime_explanation(instance, top_n)
            top = set(f.split(' ')[0] for f, _ in feats)
        all_top_features.append(top)

    # Jaccard similarity between all pairs
    jaccard_scores = []
    for i in range(len(all_top_features)):
        for j in range(i+1, len(all_top_features)):
            inter = len(all_top_features[i] & all_top_features[j])
            union = len(all_top_features[i] | all_top_features[j])
            jaccard_scores.append(inter / union if union > 0 else 0)
    return np.mean(jaccard_scores)

# Measure stability on 20 instances
shap_stabilities = []
lime_stabilities = []
for inst in sample_instances[:20]:
    shap_stabilities.append(measure_stability(inst, 'shap', n_runs=5))
    lime_stabilities.append(measure_stability(inst, 'lime', n_runs=5))

print('Explanation Stability (Jaccard similarity, 1.0 = perfectly stable):')
print(f'  SHAP: {np.mean(shap_stabilities):.4f} (std: {np.std(shap_stabilities):.4f})')
print(f'  LIME: {np.mean(lime_stabilities):.4f} (std: {np.std(lime_stabilities):.4f})')

### Step 6: Metric 3 — Explanation Coverage

What percentage of recommendations can each method explain?

In [ ]:
# Coverage: can we generate a meaningful explanation?
np.random.seed(42)
test_users = np.random.choice(user_ids, size=50, replace=False)
coverage = {'SHAP': 0, 'LIME': 0, 'Item-CF': 0, 'User-CF': 0}
total_tests = 0

for uid in test_users:
    if uid not in uid_map: continue
    uid_idx = uid_map[uid]
    unrated = np.where(ui_dense[uid_idx] == 0)[0]
    if len(unrated) == 0: continue
    sample_mids = np.random.choice(unrated, size=min(5, len(unrated)), replace=False)

    for mid_idx in sample_mids:
        total_tests += 1
        mid = idx_to_mid[mid_idx]

        # SHAP/LIME: always available if we have features
        if mid in movie_feat.index and uid in user_feat.index:
            coverage['SHAP'] += 1
            coverage['LIME'] += 1

        # Item-CF: need similar items user has rated
        if get_item_cf_explanation(uid, mid, k=1):
            coverage['Item-CF'] += 1

        # User-CF: need similar users who rated this
        if get_user_cf_explanation(uid, mid, k=1):
            coverage['User-CF'] += 1

print(f'Explanation Coverage ({total_tests} test pairs):')
for method, count in coverage.items():
    print(f'  {method}: {count}/{total_tests} ({count/total_tests*100:.1f}%)')

### Step 7: Metric 4 — Computational Cost

In [ ]:
# Measure time per explanation
instance = X_test[0]
uid_test, mid_test = 1, 260  # User 1, Star Wars
n_timing = 20

# SHAP timing
start = time.time()
for _ in range(n_timing):
    get_shap_explanation(instance)
shap_time = (time.time() - start) / n_timing

# LIME timing
start = time.time()
for _ in range(n_timing):
    get_lime_explanation(instance)
lime_time = (time.time() - start) / n_timing

# Item-CF timing
start = time.time()
for _ in range(n_timing):
    get_item_cf_explanation(uid_test, mid_test)
item_cf_time = (time.time() - start) / n_timing

# User-CF timing
start = time.time()
for _ in range(n_timing):
    get_user_cf_explanation(uid_test, mid_test)
user_cf_time = (time.time() - start) / n_timing

print('Computational Cost (avg time per explanation):')
print(f'  SHAP:    {shap_time*1000:>8.2f} ms')
print(f'  LIME:    {lime_time*1000:>8.2f} ms')
print(f'  Item-CF: {item_cf_time*1000:>8.2f} ms')
print(f'  User-CF: {user_cf_time*1000:>8.2f} ms')

### Step 8: Metric 5 — Bias Detection

Do explanations reveal biases in recommendations? Check if certain genres or popularity levels are over-represented.

In [ ]:
# Analyze SHAP values to detect feature biases
shap_sample = X_test[np.random.choice(len(X_test), 300, replace=False)]
shap_vals = shap_explainer.shap_values(shap_sample)
mean_shap = np.abs(shap_vals).mean(axis=0)
feat_importance = pd.Series(mean_shap, index=feature_names).sort_values(ascending=False)

# Check genre bias: which genres dominate explanations?
genre_importance = {}
for feat, imp in feat_importance.items():
    for g in all_genres:
        if g.lower() in feat.lower():
            genre_importance[g] = genre_importance.get(g, 0) + imp

genre_imp_sorted = sorted(genre_importance.items(), key=lambda x: -x[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genre bias
genres_plot = [g for g, _ in genre_imp_sorted[:12]]
values_plot = [v for _, v in genre_imp_sorted[:12]]
axes[0].barh(genres_plot[::-1], values_plot[::-1], color='#2196F3', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Total SHAP Importance')
axes[0].set_title('Genre Bias in Explanations')
axes[0].grid(True, alpha=0.3, axis='x')

# Popularity bias: is avg_rating dominating?
cat_importance = {'Movie Genres': 0, 'User Preferences': 0, 'Popularity/Year': 0}
for feat, imp in feat_importance.items():
    if 'movie_g_' in feat:
        cat_importance['Movie Genres'] += imp
    elif 'user_g_' in feat:
        cat_importance['User Preferences'] += imp
    else:
        cat_importance['Popularity/Year'] += imp

colors = ['#4CAF50', '#FF9800', '#F44336']
axes[1].pie(cat_importance.values(), labels=cat_importance.keys(), colors=colors,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Feature Category Distribution in Explanations')

plt.tight_layout()
plt.show()

# Popularity bias check
pop_importance = feat_importance.get('movie_avg_rating', 0) + feat_importance.get('movie_year', 0)
total_importance = feat_importance.sum()
print(f'\nPopularity/Year features account for {pop_importance/total_importance*100:.1f}% of total importance.')
if pop_importance/total_importance > 0.3:
    print('⚠️  BIAS DETECTED: Popularity and year features are disproportionately influential.')
    print('   The model may be recommending popular movies regardless of user preferences.')
else:
    print('✓ Popularity bias is within acceptable range.')

### Step 9: Summary Comparison Dashboard

In [ ]:
# Summary comparison table
print('='*75)
print('EXPLAINABILITY METHODS COMPARISON')
print('='*75)
print(f'{"Metric":<25s} | {"SHAP":>10s} | {"LIME":>10s} | {"Item-CF":>10s} | {"User-CF":>10s}')
print('-'*75)

# Fidelity
print(f'{"Fidelity (pred change)":<25s} | {shap_fidelity:>10.4f} | {lime_fidelity:>10.4f} | {"N/A":>10s} | {"N/A":>10s}')

# Stability
print(f'{"Stability (Jaccard)":<25s} | {np.mean(shap_stabilities):>10.4f} | {np.mean(lime_stabilities):>10.4f} | {"1.0000":>10s} | {"1.0000":>10s}')

# Coverage
for method in ['SHAP', 'LIME', 'Item-CF', 'User-CF']:
    pass  # already printed above
cov_vals = [coverage[m]/total_tests*100 for m in ['SHAP', 'LIME', 'Item-CF', 'User-CF']]
print(f'{"Coverage (%)":<25s} | {cov_vals[0]:>9.1f}% | {cov_vals[1]:>9.1f}% | {cov_vals[2]:>9.1f}% | {cov_vals[3]:>9.1f}%')

# Speed
times = [shap_time*1000, lime_time*1000, item_cf_time*1000, user_cf_time*1000]
print(f'{"Speed (ms)":<25s} | {times[0]:>9.1f}ms | {times[1]:>9.1f}ms | {times[2]:>9.1f}ms | {times[3]:>9.1f}ms')

# Explanation type
print(f'{"Type":<25s} | {"Feature":>10s} | {"Feature":>10s} | {"Item-based":>10s} | {"User-based":>10s}')
print(f'{"Model-Agnostic?":<25s} | {"Partial":>10s} | {"Yes":>10s} | {"Yes":>10s} | {"Yes":>10s}')
print(f'{"Bias Detection?":<25s} | {"Yes":>10s} | {"Yes":>10s} | {"Limited":>10s} | {"Limited":>10s}')

In [ ]:
# Visual summary
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
methods = ['SHAP', 'LIME', 'Item-CF', 'User-CF']
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

# Fidelity
fid_vals = [shap_fidelity, lime_fidelity, 0, 0]
axes[0,0].bar(methods, fid_vals, color=colors, edgecolor='black', alpha=0.8)
axes[0,0].set_title('Fidelity (Prediction Change)')
axes[0,0].set_ylabel('Avg |Δ prediction|')
axes[0,0].grid(True, alpha=0.3, axis='y')

# Stability
stab_vals = [np.mean(shap_stabilities), np.mean(lime_stabilities), 1.0, 1.0]
axes[0,1].bar(methods, stab_vals, color=colors, edgecolor='black', alpha=0.8)
axes[0,1].set_title('Stability (Jaccard Similarity)')
axes[0,1].set_ylabel('Avg Jaccard')
axes[0,1].set_ylim(0, 1.1)
axes[0,1].grid(True, alpha=0.3, axis='y')

# Coverage
axes[1,0].bar(methods, cov_vals, color=colors, edgecolor='black', alpha=0.8)
axes[1,0].set_title('Coverage (%)')
axes[1,0].set_ylabel('% of pairs explainable')
axes[1,0].set_ylim(0, 110)
axes[1,0].grid(True, alpha=0.3, axis='y')

# Speed (log scale)
axes[1,1].bar(methods, times, color=colors, edgecolor='black', alpha=0.8)
axes[1,1].set_title('Computational Cost')
axes[1,1].set_ylabel('Time per explanation (ms)')
axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3, axis='y')

for ax in axes.flat:
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Explainability Methods Comparison Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Analysis

**Do explanations make recommendations clearer?**

Yes, but each method offers a different type of clarity:
- **SHAP** provides precise, quantitative feature attributions — ideal for understanding *what* drives a prediction. High fidelity and stability make it reliable.
- **LIME** offers similar feature-level explanations but is model-agnostic, working on any model. Slightly less stable due to random perturbations.
- **Item-CF** gives the most intuitive explanations (*"because you liked X, Y, Z"*) — users can immediately verify the reasoning against their own experience.
- **User-CF** provides social proof (*"similar users liked this"*) but is less personal since users don't know the neighbors.

**Do explanations reveal biases?**

Yes. SHAP and LIME can detect:
- **Popularity bias**: If `avg_rating` dominates explanations, the model recommends popular movies regardless of personal taste.
- **Genre bias**: If certain genres (e.g., Drama) consistently appear as top features, the model may have a genre skew.
- **Recency bias**: If `year` is highly influential, newer movies may be unfairly favored.

Neighborhood methods reveal a different kind of bias:
- **Filter bubble**: If Item-CF always points to the same few similar items, the user is stuck in a recommendation loop.
- **Cold start**: Low coverage in User-CF/Item-CF for new users or niche movies reveals where the system fails.

**Recommendation**: Use SHAP for model debugging and bias detection, Item-CF for user-facing explanations, and LIME as a fallback for black-box models.